# 9. Fine-tuning amb SetFit

Tractament de Dades Textuals i Codificades — Eixample Clínic, 2026

Pol Pastells, ppastells@eixampleclinic.es


## Objectius

En aquesta sessió aprendrem a **adaptar un model preentrenat** a la nostra tasca clínica específica usant **pocs exemples etiquetats**.

A la sessió anterior vam veure que el zero-shot funcionava... regular. Avui ho millorarem.

#### Tasca: Detecció d'exacerbació de MPOC en notes clíniques (binària)

## 1. D'on venim?

A la sessió anterior vam:
- Generar **embeddings** de notes clíniques amb sentence-transformers
- Construir un **cercador semàntic**
- Provar **zero-shot classification** per detectar exacerbacions
- Veure que l'accuracy era... millorable 😬

**Pregunta clau**: tenim ~79 notes etiquetades com a "exacerbació". Com les aprofitem?

### Opcions:
| Tècnica | Dades necessàries | Cost computacional | Quan usar-la |
|---------|-------------------|--------------------|--------------| 
| Zero-shot | 0 exemples | Baix (només inferència) | Prototipatge ràpid |
| **SetFit** | **8-64 per classe** | **Baix-mitjà (CPU)** | **Pocs exemples, dataset petit** |
| Fine-tuning clàssic | 100s-1000s | Alt (sovint GPU) | Dataset gran, màxim rendiment |
| LLM + prompt | 0-pocs | Molt alt (API o GPU gran) | Tasques complexes/generatives |

Avui: **SetFit**. La propera classe: fine-tuning clàssic.


In [ ]:
# !pip install -U "setfit>=1.1.0" "transformers>=4.41,<5.0" sentence-transformers scikit-learn matplotlib seaborn pandas -q

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, models
from setfit import SetFitModel
from sklearn.decomposition import PCA
from transformers import pipeline

warnings.filterwarnings("ignore")

from datasets import Dataset
from sentence_transformers import SentenceTransformer
from setfit import SetFitModel, Trainer, TrainingArguments
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

colors_grup = {
    "exacerbació": "#e74c3c",
    "estable": "#3498db",
    "no_etiquetat": "#95a5a6",
}
SEED = 42
np.random.seed(SEED)

## 2. Què és SetFit i per què el fem servir?

### El problema del fine-tuning clàssic
Per fer fine-tuning "tradicional" d'un model BERT necessites:
- Centenars o milers d'exemples etiquetats
- GPU (en CPU pot trigar hores)
- Ajustar molts hiperparàmetres (learning rate, batch size, epochs...)

### La solució SetFit
**SetFit** (Sentence Transformer Fine-tuning) funciona en 2 passos:

1. **Fine-tuning contrastiu** dels embeddings:
   - Crea parells de frases (mateixa classe = positius, diferent classe = negatius)
   - Ajusta el model perquè frases similars *en la nostra tasca* tinguin embeddings més propers
   
2. **Classificador clàssic** (regressió logística) sobre els embeddings ajustats

### Avantatges
- ✅ Funciona amb **8-64 exemples per classe**
- ✅ Ràpid en CPU
- ✅ No cal escriure prompts ni format especial
- ✅ Resultats sovint comparables al fine-tuning complet

### Referència
Tunstall et al. (2022) — *Efficient Few-Shot Learning Without Prompts*: https://arxiv.org/abs/2209.11055

## 3. Carreguem i preparem les dades

Reutilitzem el dataset de notes clíniques de la sessió anterior.


In [ ]:
notes = pd.read_csv("data/CLASSIFICACIÓ_NOTES_CLINIQUES.csv", sep=";")
test = notes[notes.exacerbacio.isna()].copy()
df = notes[~notes.exacerbacio.isna()].copy()

df.exacerbacio = df.exacerbacio.map({"SI": 1, "si": 1, "NO": 0})
df.idioma_dominant = df.idioma_dominant.map({"CATALÀ": "ca", "ESPANYOL": "es"})
assert df.idioma_dominant.isna().sum() == 0

for col in ("text", "motiu", "exploracio", "avaluacio", "pla"):
    df[col] = df[col].str.lower().str.replace("\n", " ")

print(f"Total notes: {len(df)}")
print("\nDistribució de classes:")
print(df["exacerbacio"].value_counts())

df.head(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["exacerbacio"].value_counts().plot(kind="bar", ax=ax, color=["steelblue", "tomato"])
ax.set_xticklabels(["No exacerbació (0)", "Exacerbació (1)"], rotation=0)
ax.set_ylabel("Nombre de notes")
ax.set_title("Distribució de classes — clarament desbalancejat")
for i, v in enumerate(df["exacerbacio"].value_counts()):
    ax.text(i, v + 5, str(v), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print("Dataset desbalancejat → l'accuracy NO és prou bona mètrica!")
print("Usarem F1-score, precision i recall.")

### Split estratificat: train / validation / test

- **Train**: per entrenar SetFit (en farem servir només uns pocs exemples)
- **Validation**: per triar hiperparàmetres / model base
- **Test**: per l'avaluació final (només l'usem un cop!)

L'**estratificació** garanteix que la proporció de classes es manté en cada split.


In [ ]:
# Split inicial: 70% train, 15% val, 15% test
df_train, df_temp = train_test_split(df, test_size=0.30, stratify=df["exacerbacio"], random_state=SEED)
df_val, df_test = train_test_split(df_temp, test_size=0.50, stratify=df_temp["exacerbacio"], random_state=SEED)
y_test = df_test.exacerbacio.to_numpy()

print(f"Train: {len(df_train)}  ({df_train['exacerbacio'].sum()} positius)")
print(f"Val:   {len(df_val)}    ({df_val['exacerbacio'].sum()} positius)")
print(f"Test:  {len(df_test)}   ({df_test['exacerbacio'].sum()} positius)")

### Few-shot: agafem només N exemples per classe

La gràcia de SetFit és funcionar amb **pocs exemples**. Anem a simular un escenari realista:
"acabem d'etiquetar 16 notes positives i 16 negatives, podem fer un classificador útil?"


In [ ]:
N_PER_CLASS = 16


def sample_few_shot(df, n_per_class, seed=SEED):
    """Agafa n_per_class exemples de cada classe."""
    pos = df[df["exacerbacio"] == 1].sample(n=n_per_class, random_state=seed)
    neg = df[df["exacerbacio"] == 0].sample(n=n_per_class, random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed).reset_index(drop=True)


df_train_fs = sample_few_shot(df_train, N_PER_CLASS)
print(f"Train few-shot: {len(df_train_fs)} exemples ({N_PER_CLASS} per classe)")
df_train_fs.head()

In [ ]:
# SetFit espera columnes "text" i "label"
def to_hf_dataset(df, text_col="text", label_col="exacerbacio"):
    return Dataset.from_pandas(df[[text_col, label_col]].rename(columns={label_col: "label"}), preserve_index=False)


ds_train = to_hf_dataset(df_train_fs)
ds_val = to_hf_dataset(df_val)
ds_test = to_hf_dataset(df_test)

print(ds_train)

## 4. Baseline: recordem el zero-shot

Abans d'entrenar res, recuperem les prediccions del zero-shot de la sessió anterior com a **referència**.


In [ ]:
zs = pipeline(
    "zero-shot-classification",
    # model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7",
    model="joeddav/xlm-roberta-large-xnli",  # Multilingüe
)

candidate_labels = ["exacerbació de MPOC", "estat estable de MPOC"]

print("Executant zero-shot sobre el test set...")
zs_preds = []
for text in ds_test["text"]:
    out = zs(text, candidate_labels=candidate_labels)
    pred = 1 if out["labels"][0] == "exacerbació de MPOC" else 0
    zs_preds.append(pred)

zs_preds = np.array(zs_preds)

print("\n--- Zero-shot baseline ---")
print(classification_report(y_test, zs_preds, target_names=["No exac.", "Exac."]))

In [ ]:
import gc

import torch


def cleanup():
    """Funció per borrar models de gpu"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        torch.cuda.synchronize()

In [ ]:
del zs
cleanup()

## 5. Primer entrenament amb SetFit

### Triem un model base
Comencem amb un model multilingüe petit i ràpid: `paraphrase-multilingual-MiniLM-L12-v2`.

Després el comparem amb altres.


In [ ]:
args = TrainingArguments(
    batch_size=16,
    num_epochs=1,  # SetFit ja fa moltes iteracions internament
    # amb num_iterations més alt millora una mica però triga més
    num_iterations=20,  # parells contrastius generats per exemple
    # tipus de dades, és típic fer servir fp16, dependrà de la GPU,
    # amb setfit aquest paràmetre = fp16
    use_amp=True,
)

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SetFitModel.from_pretrained(MODEL_NAME)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    metric="f1",
)

trainer.train()

In [ ]:
# Avaluació al validation set
metrics_val = trainer.evaluate(ds_val)
print(f"F1 al validation: {metrics_val['f1']:.3f}")

# Prediccions al test
y_pred = model.predict(ds_test["text"])
y_pred = np.array(y_pred)

print("\n--- SetFit ---")
print(classification_report(y_test, y_pred, target_names=["No exac.", "Exac."]))
minilm_metrics = {
    "model": "paraphrase-multilingual-MiniLM-L12-v2",
    "f1": f1_score(y_test, y_pred),
    "accuracy": accuracy_score(y_test, y_pred),
}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, preds, title in zip(axes, [zs_preds, y_pred], ["Zero-shot", "SetFit (16 ex/classe)"]):
    cm = confusion_matrix(y_test, preds, labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["No exac.", "Exac."])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    f1 = f1_score(y_test, preds)
    acc = accuracy_score(y_test, preds)
    ax.set_title(f"{title}\nF1={f1:.2f} | Acc={acc:.2f}")

plt.tight_layout()
plt.show()

## 6. Què ha après SetFit? Visualitzem els embeddings

Comparem els embeddings **abans** i **després** del fine-tuning amb PCA.

Si SetFit ha funcionat, hauríem de veure que les classes se separen millor a l'espai d'embeddings després de l'entrenament.


In [ ]:
# Embeddings abans (model original)
model_original = SentenceTransformer(MODEL_NAME)
emb_before = model_original.encode(ds_test["text"], show_progress_bar=False)

# Embeddings després (model fine-tuned)
emb_after = model.model_body.encode(ds_test["text"], show_progress_bar=False)

# PCA (un reductor per cada espai d'embeddings)
pca_before = PCA(n_components=2)
emb_before_2d = pca_before.fit_transform(emb_before)

pca_after = PCA(n_components=2)
emb_after_2d = pca_after.fit_transform(emb_after)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, emb_2d, pca, title in zip(
    axes,
    [emb_before_2d, emb_after_2d],
    [pca_before, pca_after],
    ["Abans (model original)", "Després (SetFit fine-tuned)"],
):
    for label, color, name in [(0, "steelblue", "No exac."), (1, "tomato", "Exac.")]:
        mask = y_test == label
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], c=color, label=name, alpha=0.6, s=30)
    var_exp = sum(pca.explained_variance_ratio_) * 100
    ax.set_title(f"{title}\n(variància explicada: {var_exp:.1f}%)")
    ax.legend()
    ax.set_xlabel("PC 1")
    ax.set_ylabel("PC 2")

plt.tight_layout()
plt.show()

In [ ]:
del model
del trainer
cleanup()

## 7. Comparem diversos models base

Provem 2-3 models base diferents per veure quin va millor a la nostra tasca.

⚠️ **Compte**: això pot trigar


In [ ]:
MODELS = [
    # "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",  # ja fet a dalt
    "intfloat/multilingual-e5-small",  # petit
    "sentence-transformers/paraphrase-multilingual-mpnet-base-v2",  # millor però més lent
]

try:
    results = [minilm_metrics]
except:
    results = []


for model_name in MODELS:
    print(f"\n🔄 Entrenant {model_name.split('/')[-1]}...")
    if "SapBERT" in model_name:
        m = sapbert(model_name)
    else:
        m = SetFitModel.from_pretrained(model_name)
    t = Trainer(
        model=m,
        args=args,
        train_dataset=ds_train,
        eval_dataset=ds_val,
        metric="f1",
    )
    t.train()
    preds = np.array(m.predict(ds_test["text"]))
    results.append(
        {
            "model": model_name.split("/")[-1],
            "f1": f1_score(y_test, preds),
            "accuracy": accuracy_score(y_test, preds),
        }
    )

df_results = pd.DataFrame(results)
print("\n--- Comparació final ---")
print(df_results.to_string(index=False))